# ⚔️ MOTUS-VIGILUS — Frégate U-GAMMA : La Forge
> Manifestation : Données .npz → Animation .fbx (Roblox R15)

**Pipeline** : 📦 .npz → [Blender Headless] → [Rigging R15] → [Bake Animation] → 🎮 .fbx

**Stockage** : Tout est sur Google Drive (`MOTUS-VIGILUS/U-GAMMA/`)

**Instructions** :
1. Exécuter l'installation (monte Drive + installe Blender)
2. Déposer vos fichiers .npz dans `Drive/MOTUS-VIGILUS/U-GAMMA/inputs/`
3. Lancer la forge
4. Récupérer les .fbx dans `Drive/MOTUS-VIGILUS/U-GAMMA/outputs/`
5. Importer dans Roblox Studio

In [ ]:
# ═══════ INSTALLATION + MONTAGE DRIVE — FRÉGATE U-GAMMA ═══════
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = "/content/drive/My Drive/MOTUS-VIGILUS/U-GAMMA"
INPUTS_DIR = f"{DRIVE_BASE}/inputs"
OUTPUTS_DIR = f"{DRIVE_BASE}/outputs"
TEMPLATES_DIR = f"{DRIVE_BASE}/templates"

for d in [INPUTS_DIR, OUTPUTS_DIR, TEMPLATES_DIR]:
    os.makedirs(d, exist_ok=True)

# Installer les dépendances (UNIQUEMENT numpy + blender — étanchéité)
!pip install -q numpy
import os, subprocess

BLENDER_VERSION = "4.2.0"
BLENDER_URL = f"https://download.blender.org/release/Blender4.2/blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BLENDER_DIR = f"/content/blender-{BLENDER_VERSION}-linux-x64"
BLENDER_BIN = f"{BLENDER_DIR}/blender"

if not os.path.exists(BLENDER_BIN):
    print(f"⬇️ Téléchargement Blender {BLENDER_VERSION}...")
    subprocess.run(["wget", "-q", "-O", "/tmp/blender.tar.xz", BLENDER_URL], check=True)
    subprocess.run(["tar", "-xf", "/tmp/blender.tar.xz", "-C", "/content/"], check=True)
    os.remove("/tmp/blender.tar.xz")
    print(f"✅ Blender {BLENDER_VERSION} installé : {BLENDER_BIN}")
else:
    print(f"✅ Blender {BLENDER_VERSION} déjà présent")

# Cloner le codebase
REPO_DIR = "/content/MOTUS-VIGILUS"
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull -q
else:
    !git clone -q https://github.com/kioka8877-ux/MOTUS-VIGILUS..git {REPO_DIR}

# Cacher le template R15 sur Drive (1 fois, 39MB)
import shutil
TEMPLATE_SRC = f"{REPO_DIR}/U-GAMMA/templates/r15_template.blend"
TEMPLATE_DRIVE = f"{TEMPLATES_DIR}/r15_template.blend"
if not os.path.exists(TEMPLATE_DRIVE) and os.path.exists(TEMPLATE_SRC):
    shutil.copy2(TEMPLATE_SRC, TEMPLATE_DRIVE)
    print("✅ Template R15 caché sur Drive")
elif os.path.exists(TEMPLATE_DRIVE):
    print("✅ Template R15 chargé depuis Drive")

# Utiliser le template depuis Drive
TEMPLATE_PATH = TEMPLATE_DRIVE if os.path.exists(TEMPLATE_DRIVE) else TEMPLATE_SRC

print(f"✅ Frégate U-GAMMA — Installation terminée")
print(f"📁 Inputs    : {INPUTS_DIR}")
print(f"📁 Outputs   : {OUTPUTS_DIR}")
print(f"📁 Template  : {TEMPLATE_PATH}")

In [ ]:
# ═══════ VÉRIFICATION INPUTS ═══════
import glob

npz_files = sorted(glob.glob(f"{INPUTS_DIR}/motus_core_P*.npz"))

if not npz_files:
    print(f"❌ Aucun fichier .npz trouvé dans {INPUTS_DIR}")
    print("   Copiez les .npz produits par la Frégate U-ALPHA dans ce dossier Drive")
    print("   puis relancez cette cellule")
else:
    print(f"📦 {len(npz_files)} fichier(s) .npz trouvés :")
    for f in npz_files:
        size_kb = os.path.getsize(f) / 1024
        print(f"  ✅ {os.path.basename(f)} ({size_kb:.1f} KB)")

In [ ]:
# ═══════ LA FORGE — FRÉGATE U-GAMMA ═══════
import glob, os

BLENDER_VERSION = "4.2.0"
BLENDER_BIN = f"/content/blender-{BLENDER_VERSION}-linux-x64/blender"

npz_files = sorted(glob.glob(f"{INPUTS_DIR}/motus_core_P*.npz"))
if not npz_files:
    print("❌ Aucun fichier .npz. Relancez la cellule précédente.")
else:
    FORGE_SCRIPT = f"{REPO_DIR}/U-GAMMA/codebase/motus_forge.py"
    LOCAL_OUT = "/content/gamma_outputs"
    os.makedirs(LOCAL_OUT, exist_ok=True)
    
    fbx_files = []
    for npz in npz_files:
        fbx_name = os.path.basename(npz).replace(".npz", ".fbx").replace("motus_core_", "MOTUS_VIGILUS_")
        fbx_out = f"{LOCAL_OUT}/{fbx_name}"
        print(f"\n🔥 Forge : {os.path.basename(npz)} → {fbx_name}")
        !{BLENDER_BIN} -b "{TEMPLATE_PATH}" -P "{FORGE_SCRIPT}" -- "{npz}" "{fbx_out}"
        
        # Copier vers Drive
        import shutil
        drive_out = f"{OUTPUTS_DIR}/{fbx_name}"
        shutil.copy2(fbx_out, drive_out)
        fbx_files.append(drive_out)

    print(f"\n✅ {len(fbx_files)} fichier(s) .fbx forgés et sauvegardés sur Drive :")
    for f in fbx_files:
        print(f"  🎮 {os.path.basename(f)}")

In [ ]:
# ═══════ VÉRIFICATION OUTPUTS ═══════
import glob, os

fbx_on_drive = sorted(glob.glob(f"{OUTPUTS_DIR}/MOTUS_VIGILUS_P*.fbx"))
print(f"🎮 {len(fbx_on_drive)} fichier(s) .fbx sur Drive :")
for f in fbx_on_drive:
    size_kb = os.path.getsize(f) / 1024
    print(f"  ✅ {os.path.basename(f)} ({size_kb:.1f} KB)")

print(f"\n⚔️ Frégate U-GAMMA — Forge terminée.")
print(f"📋 Récupérez les .fbx depuis : {OUTPUTS_DIR}")
print(f"   → Importer dans Roblox Studio → Avatar Animation Editor")